# Model Compression

## Learning Objectives
1. Understand magnitude-based weight pruning from first principles.
2. Apply structured and unstructured pruning with torch.nn.utils.prune.
3. Quantise a BERT-style model to INT8 using post-training quantisation (PTQ).
4. Analyse the accuracy-vs-compression trade-off across pruning and quantisation.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: Magnitude Pruning (NumPy)


In [ ]:
def magnitude_prune(weights: np.ndarray, sparsity: float) -> np.ndarray:
    """Zero out the smallest-magnitude weights to achieve target sparsity."""
    threshold = np.percentile(np.abs(weights), sparsity * 100)
    mask = np.abs(weights) >= threshold
    return weights * mask


def compute_sparsity(weights: np.ndarray) -> float:
    return float((weights == 0).mean())


W = np.random.randn(256, 512).astype(np.float32)
print(f"Original sparsity: {compute_sparsity(W):.3f}")
print(f"Original L2 norm : {np.linalg.norm(W):.3f}")

sparsity_levels = [0.0, 0.3, 0.5, 0.7, 0.9, 0.95]
print(f"\n{'Sparsity':>10} | {'Zeros %':>8} | {'L2 norm':>8} | {'Top-1 weight':>12}")
print("-" * 50)
for s in sparsity_levels:
    W_pruned = magnitude_prune(W, s)
    actual_sparse = compute_sparsity(W_pruned)
    l2 = np.linalg.norm(W_pruned)
    top1 = np.abs(W_pruned).max()
    print(f"{s:>10.0%} | {actual_sparse:>8.3f} | {l2:>8.3f} | {top1:>12.4f}")

W_imp = W.copy()
for round_idx in range(5):
    W_imp = magnitude_prune(W_imp, 0.1)
    print(f"IMP round {round_idx+1}: sparsity={compute_sparsity(W_imp):.3f}")


## Level 2: torch.nn.utils.prune + Dynamic Quantisation (OOM Handling)


In [ ]:
class SmallMLP(nn.Module):
    """Simple MLP for pruning and quantisation experiments."""
    def __init__(self, in_dim=128, hidden=256, out_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden, hidden)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc3(self.relu2(self.fc2(self.relu1(self.fc1(x)))))


def count_params(model: nn.Module):
    """Count total and non-zero parameters."""
    total = sum(p.numel() for p in model.parameters())
    nonzero = sum((p != 0).sum().item() for p in model.parameters())
    return total, nonzero


def model_size_mb(model: nn.Module) -> float:
    """Approximate model size in MB."""
    return sum(p.nbytes for p in model.parameters()) / 1e6


model_prune = SmallMLP().to(device)
total, nz = count_params(model_prune)
print(f"Before pruning: {total:,} params, {nz:,} non-zero, {model_size_mb(model_prune):.2f} MB")

try:
    for module in model_prune.modules():
        if isinstance(module, nn.Linear):
            prune.l1_unstructured(module, name='weight', amount=0.5)
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print("OOM: reduce model size"); torch.cuda.empty_cache()
    else:
        raise

for name, module in model_prune.named_modules():
    if isinstance(module, nn.Linear):
        mask = module.weight_mask
        sparsity_val = 1.0 - mask.float().mean().item()
        print(f"Layer {name}: {sparsity_val:.2%} sparse")

for module in model_prune.modules():
    if isinstance(module, nn.Linear):
        prune.remove(module, 'weight')

total_p, nz_p = count_params(model_prune)
print(f"After pruning:  {total_p:,} params, {nz_p:,} non-zero ({100*nz_p/total_p:.1f}% non-zero)")

model_quantised = torch.quantization.quantize_dynamic(
    SmallMLP(), {nn.Linear}, dtype=torch.qint8,
)
print(f"\nQuantised model size: {model_size_mb(model_quantised):.3f} MB")
print(f"Original  model size: {model_size_mb(SmallMLP()):.3f} MB")
print(f"Compression ratio   : {model_size_mb(SmallMLP())/model_size_mb(model_quantised):.2f}x")


## Real-World Example 1: Post-Training Quantisation on a BERT-Style Encoder


In [ ]:
import time

class BERTLikeEncoder(nn.Module):
    """Toy transformer encoder mimicking BERT architecture."""
    def __init__(self, vocab_size=1000, d_model=256, nhead=4, n_layers=3, n_classes=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=512,
            batch_first=True, dropout=0.1,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        enc = self.encoder(emb)
        return self.classifier(enc[:, 0, :])


bert_model = BERTLikeEncoder()
bert_model.eval()

bert_quantised = torch.quantization.quantize_dynamic(
    bert_model, {nn.Linear}, dtype=torch.qint8
)

dummy_input = torch.randint(0, 1000, (8, 64))


def measure_latency(model, input_ids, n_runs=30):
    times = []
    model.eval()
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = model(input_ids)
            times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times[5:])


lat_fp32 = measure_latency(bert_model, dummy_input)
lat_int8 = measure_latency(bert_quantised, dummy_input)

size_fp32 = sum(p.nbytes for p in bert_model.parameters()) / 1e6
size_int8 = sum(p.nbytes for p in bert_quantised.parameters()) / 1e6

print(f"FP32 model size: {size_fp32:.2f} MB  | latency: {lat_fp32:.2f} ms")
print(f"INT8 model size: {size_int8:.2f} MB  | latency: {lat_int8:.2f} ms")
print(f"Size reduction : {size_fp32/size_int8:.2f}x")
print(f"Latency speedup: {lat_fp32/lat_int8:.2f}x")


## Real-World Example 2: Knowledge Distillation + Pruning Combined


In [ ]:
class TeacherNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 5))
    def forward(self, x): return self.net(x)


class StudentNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 5))
    def forward(self, x): return self.net(x)


teacher = TeacherNet().to(device).eval()
student = StudentNet().to(device)

X_kd = torch.randn(1000, 64); y_kd = torch.randint(0, 5, (1000,))
kd_loader = DataLoader(TensorDataset(X_kd, y_kd), batch_size=64, shuffle=True)

opt_kd = torch.optim.Adam(student.parameters(), lr=1e-3)
ce_loss = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction='batchmean')
TEMPERATURE = 4.0
ALPHA = 0.7

for epoch in range(30):
    student.train()
    total_loss = 0.0
    for xb, yb in kd_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt_kd.zero_grad()
        try:
            with torch.no_grad():
                teacher_logits = teacher(xb) / TEMPERATURE
                soft_targets = torch.softmax(teacher_logits, dim=-1)
            student_logits = student(xb)
            student_soft = torch.log_softmax(student_logits / TEMPERATURE, dim=-1)
            loss_hard = ce_loss(student_logits, yb)
            loss_soft = kl_loss(student_soft, soft_targets) * (TEMPERATURE ** 2)
            loss = ALPHA * loss_soft + (1 - ALPHA) * loss_hard
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("OOM"); torch.cuda.empty_cache(); continue
            raise
        loss.backward(); opt_kd.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} | KD loss: {total_loss/len(kd_loader):.4f}")

for module in student.modules():
    if isinstance(module, nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.50)
        prune.remove(module, 'weight')

t_sz = sum(p.nbytes for p in TeacherNet().parameters()) / 1e6
s_sz = sum(p.nbytes for p in student.parameters()) / 1e6
nz_ratio = sum((p != 0).sum().item() for p in student.parameters()) /            sum(p.numel() for p in student.parameters())
print(f"Teacher size: {t_sz:.3f} MB | Student pruned size: {s_sz:.3f} MB")
print(f"Student non-zero params: {nz_ratio:.2%}")
print(f"Total compression: {t_sz / (s_sz * nz_ratio):.2f}x (size * density)")


## Real-World Example 3: Compression Trade-off Plot


In [ ]:
def train_and_evaluate(sparsity: float, n_epochs: int = 20):
    """Train a fresh MLP, prune to sparsity, return (test_acc, model_size_mb)."""
    X = torch.randn(2000, 64); y = torch.randint(0, 5, (2000,))
    loader_tr = DataLoader(TensorDataset(X[:1600], y[:1600]), batch_size=64, shuffle=True)
    X_test, y_test = X[1600:].to(device), y[1600:].to(device)

    m = nn.Sequential(
        nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 5)
    ).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    for _ in range(n_epochs):
        m.train()
        for xb, yb in loader_tr:
            opt.zero_grad()
            crit(m(xb.to(device)), yb.to(device)).backward()
            opt.step()

    if sparsity > 0:
        for module in m.modules():
            if isinstance(module, nn.Linear):
                prune.l1_unstructured(module, name='weight', amount=sparsity)
                prune.remove(module, 'weight')

    m.eval()
    with torch.no_grad():
        acc = (m(X_test).argmax(1) == y_test).float().mean().item()
    sz = sum(p.nbytes for p in m.parameters()) / 1e6
    return acc, sz


sparsities = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95]
accs, sizes = [], []
for s in sparsities:
    acc, sz = train_and_evaluate(s)
    accs.append(acc); sizes.append(sz)
    print(f"Sparsity {s:.0%} | acc: {acc:.3f} | size: {sz:.3f} MB")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot([s * 100 for s in sparsities], accs, marker='o', color='steelblue')
ax1.set_xlabel("Sparsity (%)"); ax1.set_ylabel("Test Accuracy")
ax1.set_title("Accuracy vs Pruning Sparsity"); ax1.grid(True, alpha=0.3)
ax2.plot([s * 100 for s in sparsities], sizes, marker='s', color='coral')
ax2.set_xlabel("Sparsity (%)"); ax2.set_ylabel("Model Size (MB)")
ax2.set_title("Size vs Pruning Sparsity"); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/compression_tradeoff.png", dpi=80)
plt.close()
print("Compression trade-off plot saved to /tmp/compression_tradeoff.png")
